# Multilingual Cross-Template Key-Value Extraction — Kaggle XFUND Run

**IE 643 Course Project** · Scaled Architecture Training on XFUND Benchmark

### Key Improvements in this Run:
1. **Dataset Scale-Up (XFUND 5 Latin Languages):**
   - Overcomes FUNSD data starvation by expanding from 119 forms to **~745 training forms** and **~250 test forms** across Spanish (`es`), French (`fr`), German (`de`), Italian (`it`), and Portuguese (`pt`).
2. **Canonical Focal Loss Normalization ($N_{\text{pos}}$):**
   - Normalizes focal loss by positive links rather than all $O(N^2)$ candidate pairs, preventing positive gradient dilution.
3. **One-to-One Value Constraint (`decode_one_to_one: true`):**
   - Enforces column-level argmax matching per value entity, eliminating multi-candidate collisions.
4. **Decoupled 3-Stage Curriculum with Early Stopping (Patience = 3):**
   - Phase 1: Pure Token Classification warmup (validated on Entity F1).
   - Phase 2: Teacher-Forced Relation Head warmup (validated on Linking F1).
   - Phase 3: Joint fine-tuning (validated on combined score).

> **Hardware**: Kaggle GPU (Tesla P100 16 GB or 2xT4). Expected runtime: ~30-40 minutes for 3 seeds.

## 0 · Environment Setup & Git Clone

In [ ]:
# ── Clone or Pull KeyValue-Extraction Repository ─────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/ABasakEE/KeyValue-Extraction.git"
REPO_DIR = "/kaggle/working/KeyValue-Extraction"

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"{REPO_DIR} already exists, pulling latest main branch...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Add src/ to sys.path
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"sys.path: {sys.path[:2]}")

In [ ]:
# ── Install Required Dependencies ───────────────────────────────────────
!pip install -q --upgrade pip setuptools wheel
!pip install -q seqeval faker

In [ ]:
# ── Verify GPU and Environment ──────────────────────────────────────────
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Fine-tuning on CPU will be slow.")

## 1 · Download Official XFUND Release Files

Downloads official XFUND v1.0 JSON release archives for Spanish (`es`), French (`fr`), German (`de`), Italian (`it`), and Portuguese (`pt`).

In [ ]:
import os, urllib.request

xfund_dir = "data/xfund"
os.makedirs(xfund_dir, exist_ok=True)

BASE_URL = "https://github.com/doc-analysis/XFUND/releases/download/v1.0/"
LANGUAGES = ["es", "fr", "de", "it", "pt"]

print(f"Downloading XFUND files for languages: {LANGUAGES}...")
for lang in LANGUAGES:
    for split in ["train", "val"]:
        filename = f"{lang}.{split}.json"
        target_path = os.path.join(xfund_dir, filename)
        if not os.path.isfile(target_path) or os.path.getsize(target_path) < 1000:
            url = BASE_URL + filename
            print(f"  Downloading {filename} from {url}...")
            urllib.request.urlretrieve(url, target_path)
        print(f"  [OK] {filename} ({os.path.getsize(target_path) / 1024:.1f} KB)")

print("\nXFUND download complete!")

## 2 · Smoke Test Dataset Loader & Model Architecture

In [ ]:
# Verify XFUND dataset loader parsing and sample integrity
from src.data.xfund_loader import load_xfund_dataset, summarize

train_ex, val_ex, test_ex = load_xfund_dataset("data/xfund", languages=["es", "fr", "de", "it", "pt"])
print("\n--- XFUND Corpus Summary ---")
print(f"Train Documents: {len(train_ex)} | Entities: {sum(len(e.entities) for e in train_ex)} | Links: {sum(len(e.links) for e in train_ex)}")
print(f"Val Documents:   {len(val_ex)} | Entities: {sum(len(e.entities) for e in val_ex)} | Links: {sum(len(e.links) for e in val_ex)}")
print(f"Test Documents:  {len(test_ex)} | Entities: {sum(len(e.entities) for e in test_ex)} | Links: {sum(len(e.links) for e in test_ex)}")

# Run unit smoke tests on the relation linking head
!python tests/test_linking_head.py

---
## 3 · Train Joint Extractor on Scaled XFUND Benchmark

Trains LiLT on 5 Latin-alphabet XFUND languages (~745 train forms, ~250 test forms) with:
- Decoupled 3-stage curriculum with early stopping
- Normalized focal loss ($N_{\text{pos}}$)
- One-to-one argmax matching constraint (`decode_one_to_one: true`)

In [ ]:
!python src/train.py --config configs/xfund_joint_lilt.yaml

## 4 · Inspect Evaluation Metrics

In [ ]:
import json
from pathlib import Path

metrics_file = Path("results/xfund_joint_lilt/metrics.json")
if metrics_file.exists():
    res = json.loads(metrics_file.read_text(encoding="utf-8"))
    print("=" * 65)
    print("XFUND MULTILINGUAL JOINT EXTRACTION RESULTS")
    print("=" * 65)
    print(f"Mean Entity F1 : {res.get('mean_f1', 0):.4f} +/- {res.get('std_f1', 0):.4f}")
    print(f"Mean Linking F1: {res.get('mean_linking_f1', 0):.4f} +/- {res.get('std_linking_f1', 0):.4f}")
    print("-" * 65)
    for run in res.get("runs", []):
        seed = run.get("seed", "?")
        ent_f1 = run.get("test", {}).get("f1", 0)
        lnk_f1 = run.get("test_linking", {}).get("f1", 0)
        p = run.get("test_linking", {}).get("precision", 0)
        r = run.get("test_linking", {}).get("recall", 0)
        print(f"  Seed {seed:>2} -> Entity F1: {ent_f1:.4f} | Linking F1: {lnk_f1:.4f} (P: {p:.4f}, R: {r:.4f})")
else:
    print(f"[-] Metrics file not found at {metrics_file}")

## 5 · Package Outputs for Local Download

Packs `metrics.json`, `config.json`, and the single best checkpoint `.pt` into a downloadable zip file.

In [ ]:
import shutil
from pathlib import Path

out_dir = Path("results/xfund_joint_lilt")
best_seed_model = None
best_score = -1.0

if metrics_file.exists():
    res = json.loads(metrics_file.read_text(encoding="utf-8"))
    for run in res.get("runs", []):
        seed = run.get("seed")
        score = run.get("val_score", run.get("test", {}).get("f1", 0))
        cand = out_dir / f"seed_{seed}" / "best_model.pt"
        if cand.exists() and score > best_score:
            best_score = score
            best_seed_model = cand

if best_seed_model and best_seed_model.exists():
    target = out_dir / "best_model.pt"
    shutil.copy2(best_seed_model, target)
    print(f"Copied best checkpoint ({best_seed_model.parent.name}) to {target}")

# Create zip file for download in /kaggle/working
!cd /kaggle/working && zip -q -r /kaggle/working/kaggle_xfund_results.zip KeyValue-Extraction/results/xfund_joint_lilt -x "*.pyc" "*__pycache__*"
print("[+] Packaging complete: /kaggle/working/kaggle_xfund_results.zip")
!ls -lh /kaggle/working/kaggle_xfund_results.zip